Author: Will Blanton


# Imports

In [ ]:
import ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import itertools
import sys
from pathlib import Path

from sentence_transformers import SentenceTransformer

from gensim.models.fasttext import load_facebook_vectors
from sklearn.feature_extraction.text import CountVectorizer
from typing import List
from torch.utils.data import Dataset, DataLoader

import optuna
sys.path.insert(0, str(Path("src").resolve()))
from nyt_connections_solver.models import AttentionModel, Baseline, SetTransformer
from nyt_connections_solver.harness import SearchHarness
from nyt_connections_solver.wordnet import WordNetFeatureEncoder
from nyt_connections_solver.evaluation import evaluate_puzzles, load_puzzles

# Yellow-only graph attention experiment

This notebook is copied from `connections-graph.ipynb` to establish a narrow feasibility baseline: select the yellow group from 12-word states containing yellow plus two distractor groups. Every puzzle supplies all three pairs of distractors drawn from purple, blue, and green.

The CSV does not contain color labels. The row order confirmed in `notes/connections-categories.md` is `purple, blue, green, yellow`; this notebook uses that mapping explicitly. Node features and graph edge features are computed on the full original 16-word puzzle, then both node and edge axes are sliced to the 12-word state.

This is not a full Connections solver and does not use beam search. Its main measures are exact yellow top-1 selection, mean reciprocal rank, and the fraction of puzzle dates for which yellow is selected against all three distractor pairs.


In [ ]:
from collections import Counter
from math import log
from nyt_connections_solver.models import MultiheadAttentionBlock


class PuzzleEdgeBuilder:
    """Label-free, symmetric edge features computed on the original 16 words."""
    feature_names = ("semantic_cosine", "sense_jaccard", "rare_concept_jaccard")

    def __init__(self, wordnet_encoder, semantic_dim):
        self.wordnet_encoder = wordnet_encoder
        self.semantic_dim = semantic_dim
        self.dim = len(self.feature_names)
        self._cache = {}

    def encode(self, words, node_features):
        words = tuple(words)
        if len(words) != 16 or len(set(words)) != 16:
            raise ValueError("Edges require the original 16 distinct puzzle words")
        if words in self._cache:
            return self._cache[words]
        semantic = torch.as_tensor(node_features).detach().float().cpu()[:, :self.semantic_dim]
        if semantic.shape != (16, self.semantic_dim) or not torch.isfinite(semantic).all():
            raise ValueError("Invalid semantic features for edge construction")
        normalized = F.normalize(semantic, dim=-1)
        cosine = ((normalized @ normalized.T).clamp(-1, 1) + 1) / 2
        # A missing semantic vector supplies no similarity evidence.
        valid = semantic.norm(dim=-1) > 0
        cosine *= valid[:, None] & valid[None, :]
        edges = torch.zeros(16, 16, self.dim, dtype=torch.float32)
        edges[..., 0] = cosine

        # All candidate senses, not only the three selected for node glosses.
        senses = [set(self.wordnet_encoder._senses(word)) for word in words]
        concepts = [set().union(*(self.wordnet_encoder._ancestors(s) for s in values))
                    for values in senses]
        support = Counter(concept for values in concepts for concept in values)
        # Count each concept once per word. Ubiquitous concepts have zero weight;
        # unshared concepts contribute to the union, but never the intersection.
        weights = {concept: log(16 / count) for concept, count in support.items()}
        for i in range(16):
            for j in range(i + 1, 16):
                union = senses[i] | senses[j]
                direct = len(senses[i] & senses[j]) / len(union) if union else 0.0
                union_weight = sum(weights[c] for c in concepts[i] | concepts[j])
                shared_weight = sum(weights[c] for c in concepts[i] & concepts[j])
                rare = shared_weight / union_weight if union_weight > 0 else 0.0
                edges[i, j, 1:] = torch.tensor([direct, rare])
                edges[j, i, 1:] = edges[i, j, 1:]
        edges[torch.arange(16), torch.arange(16)] = 0
        self._cache[words] = edges.clamp(0, 1)  # (16, 16, E=3)
        return self._cache[words]


class GraphAttentionBlock(MultiheadAttentionBlock):
    def __init__(self, width, heads, dropout, edge_dim, edge_bias_scale):
        super().__init__(width, width, attn_heads=heads, dropout=dropout)
        self.edge_bias_scale = float(edge_bias_scale)
        if self.edge_bias_scale < 0:
            raise ValueError("edge_bias_scale must be nonnegative")
        self.edge_projection = nn.Linear(edge_dim, heads, bias=False)
        # Start from ordinary attention and learn which relationships to use.
        nn.init.zeros_(self.edge_projection.weight)

    def attention_bias(self, edges):
        bias = self.edge_bias_scale * torch.tanh(self.edge_projection(edges))
        # (B, N, N, H) -> (B * H, N, N), as required by MultiheadAttention.
        diagonal = torch.eye(edges.shape[1], device=edges.device, dtype=torch.bool)
        bias = bias.masked_fill(diagonal[None, :, :, None], 0)
        return bias.permute(0, 3, 1, 2).reshape(-1, edges.shape[1], edges.shape[2])

    def forward(self, x, edges):
        # x: (B, N, D); edges: (B, N, N, E)
        bias = None if self.edge_bias_scale == 0 else self.attention_bias(edges).to(x.dtype)
        attended = self.attn(x, x, x, attn_mask=bias, need_weights=False)[0]
        x = self.h_norm(x + attended)
        return self.mab_norm(self.residual_proj(x) + self.ff(x))


class GraphEncoder(nn.Module):
    def __init__(self, width, depth, heads, dropout, edge_dim, edge_bias_scale):
        super().__init__()
        self.mabs = nn.ModuleList([
            GraphAttentionBlock(width, heads, dropout, edge_dim, edge_bias_scale)
            for _ in range(depth)
        ])

    def forward(self, x, edges):
        for block in self.mabs:
            x = block(x, edges)
        return x


class GraphSetTransformer(SetTransformer):
    """Existing LTR scorer with edge-biased global attention only."""
    def __init__(self, *, edge_dim=3, edge_bias_scale=1.0, **model_config):
        super().__init__(**model_config)
        self.edge_dim = edge_dim
        original = self.full_encoder
        first = original.mabs[0]
        self.full_encoder = GraphEncoder(
            first.attn.embed_dim, len(original.mabs), first.attn.num_heads,
            first.attn.dropout, edge_dim, edge_bias_scale,
        )
        # Preserve the original initialization; scale=0 is the same model.
        self.full_encoder.load_state_dict(original.state_dict(), strict=False)

    def forward(self, x, group_idx=None, edge_features=None):
        B, N, _ = x.shape
        if edge_features is None or edge_features.shape != (B, N, N, self.edge_dim):
            raise ValueError("Expected edge_features with shape (B, N, N, edge_dim)")
        x = self.proj(self.input_norm(x))  # (B, N, D)
        x = self.full_encoder(x, edge_features)  # (B, N, D)
        x = self.grouper(x, group_idx=group_idx)  # (B, C, 4, D)
        B, C, group_size, D = x.shape
        x = x.reshape(B * C, group_size, D)
        x = self.subset_encoder(x)  # Unchanged attention within each candidate.
        x = self.subset_decoder(x).squeeze(1)  # (B * C, D)
        return self.scorer(x.reshape(B, C, D))  # (B, C, 1)


def score_batch(model, x, batch, group_idx=None):
    kwargs = {}
    if isinstance(model, GraphSetTransformer):
        kwargs["edge_features"] = batch["edge_features"].to(x.device)
    return model(x, group_idx=group_idx, **kwargs)


def load_graph_harness(weights_path, device="auto", beam_width=20, state_batch_size=4):
    """Reload this notebook's checkpoint after running imports and graph definitions.

    The CLI cannot instantiate notebook-local classes; use this loader instead.
    """
    checkpoint = torch.load(weights_path, map_location="cpu", weights_only=True)
    if checkpoint["model_class"] != "GraphSetTransformer":
        raise ValueError("Expected a GraphSetTransformer checkpoint")
    if checkpoint["edge_feature_names"] != list(PuzzleEdgeBuilder.feature_names):
        raise ValueError("Checkpoint edge feature definitions do not match")
    resolved = ("cuda" if torch.cuda.is_available() else "cpu") if device == "auto" else device
    embedder = SentenceTransformer(checkpoint["embedding_model_name"], device=str(resolved))
    encoder = WordNetFeatureEncoder(
        embedder, max_senses=checkpoint["wordnet_max_senses"],
        puzzle_conditioned=checkpoint["wordnet_puzzle_conditioned"],
    )
    config = checkpoint["model_config"]
    model = GraphSetTransformer(group_idx=torch.tensor([[0, 1, 2, 3]]), **config)
    model.load_state_dict(checkpoint["model_state_dict"])
    edges = PuzzleEdgeBuilder(encoder, config["semantic_dim"])
    harness = SearchHarness.from_model(
        model, embedder, input_dim=config["in_dim"], device=resolved,
        wordnet_encoder=encoder, edge_feature_builder=edges.encode,
        beam_width=beam_width, state_batch_size=state_batch_size,
    )
    harness.checkpoint = checkpoint
    harness.weights_path = Path(weights_path)
    return harness


In [ ]:
from pytorchltr.loss import LambdaNDCGLoss2
from pytorchltr.evaluation import ndcg

# Constants

In [ ]:
DATASET_PATH = Path("data/connections.csv")
TARGET_COLOR = "yellow"
COLORS_BY_ROW = ("purple", "blue", "green", "yellow")
TARGET_ROW = COLORS_BY_ROW.index(TARGET_COLOR)
STATE_WORDS = 12
TARGET_POSITIONS = tuple(range(8, 12))


# Helper Functions

In [ ]:
def extract_features(word):
    return pd.DataFrame({word:{
        'length': len(word),
        'num_vowels': sum(c in 'aeiou' for c in word.lower()),
        'num_consonants': sum(c.isalpha() and c not in 'aeiou' for c in word.lower()),
        'ends_with_ing': int(word.lower().endswith('ing')),
        'ends_with_ed': int(word.lower().endswith('ed')),
        'is_palindrome': int(word.lower() == word[::-1].lower())
    }})

def extract_structural_features(words):
    """
    Extract basic structural features for the given words.
    """
    return torch.tensor(pd.concat([extract_features(w) for w in words], axis=1).T.values) 

In [ ]:
def construct_yellow_labels():
    """Return every 4-word candidate and one positive: yellow at positions 8..11."""
    group_idx = torch.tensor(list(itertools.combinations(range(STATE_WORDS), 4)), dtype=torch.long)
    target = set(TARGET_POSITIONS)
    relevance = torch.tensor([int(set(candidate.tolist()) == target) for candidate in group_idx])
    return group_idx, relevance


In [ ]:
from torch.utils.data._utils.collate import default_collate


def collate_keep_wordlists(batch):
    # batch is a list of dicts
    keys = batch[0].keys()
    out = {}
    for k in keys:
        vals = [b[k] for b in batch]
        if k in {"word_list", "source_group_labels"}:
            # keep as list-of-lists grouped by sample: [ [w1..w16], ... ] (len = B)
            out[k] = vals
        else:
            out[k] = default_collate(vals)
    return out

# Create Dataset

Rerun from the top after changing `DATASET_PATH`; existing outputs may reflect an earlier dataset.

In [ ]:
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

In [ ]:
print(f"Cuda available: {torch.cuda.is_available()}")

In [ ]:
# Multilingual frozen encoder (100+ languages); D is inferred below.
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
# Use symmetric, unprompted word embeddings in both training and the harness.
word_embedder = SentenceTransformer(EMBEDDING_MODEL)
word_embedder.requires_grad_(False)
word_embedder.eval()

In [ ]:
test_emb = word_embedder.encode("Hello")

In [ ]:
D = len(test_emb)
D

In [ ]:
# ft = load_facebook_vectors('pretrained/cc.en.300.bin')

In [ ]:
class FastTextEmbedder:
    """
    Wrap gensim embeddings for simpler conversion to tensors and to fit with sentence-transformers api for easy swapping
    """
    def __init__(self, ft_model):
        self.ft = ft_model
        self.dim = ft_model.vector_size

    def encode(self, word_list, convert_to_tensor=True):
        vectors = []
        for word in word_list:
            vectors.append(self.ft[word])

        vectors = np.stack(vectors)

        if convert_to_tensor:
            return torch.tensor(vectors, dtype=torch.float)
        else:
            return vectors

In [ ]:
# word_embedder = FastTextEmbedder(ft)

In [ ]:
connections_df = pd.read_csv(DATASET_PATH, index_col=0)
# connections_df.drop(columns='category', inplace=True)

# fix issues with incorrect format... (not worth going in and adding a mechanism to correct...)
connections_df.loc[1298, "connections"] = "['line', 'plane', 'point', 'solid']"
connections_df.loc[1892, "connections"] = "['abyss', 'fly', 'matrix', 'thing']"

# remove april fools samples since they include emojis or other potentially noisy samples
connections_df = connections_df[~connections_df['date'].str.contains("04-01")]
connections_df.reset_index(drop=True, inplace=True)

connections_df['connections'] = connections_df['connections'].apply(ast.literal_eval)
connections_df.head()

In [ ]:
all_words = list(itertools.chain.from_iterable(
    connections_df
    .groupby('date')['connections']
    .agg(lambda lists: list(itertools.chain.from_iterable(lists)))
    .values
))
all_words[:10]

In [ ]:
# [w for w in all_words if ('_' in w) or ('-' in w)]

In [ ]:
# Fit after the train/validation/test split so held-out vocabulary is not learned.
vectorizer = CountVectorizer(analyzer='char', ngram_range=(1, 3))

In [ ]:
class WordFeatureBuilder:
    """
    Build word-level feature tensors for each puzzle state.

    Features are concatenated per word, so existing set/scorer models can consume
    them by changing only in_dim.
    """
    def __init__(
        self,
        word_embedder=None,
        char_vectorizer=None,
        use_semantic=True,
        use_structural=False,
        use_phonetic=False,
        use_char_svd=False,
        char_svd_dim=64,
        wordnet_encoder=None,
        use_wordnet=False,
    ):
        self.word_embedder = word_embedder
        self.char_vectorizer = char_vectorizer
        self.use_semantic = use_semantic
        self.use_structural = use_structural
        self.use_phonetic = use_phonetic
        self.use_char_svd = use_char_svd
        self.char_svd_dim = char_svd_dim
        self.wordnet_encoder = wordnet_encoder
        self.use_wordnet = use_wordnet
        self.char_svd = None
        self.struct_mean = None
        self.struct_std = None

    def fit(self, words):
        words = list(words)
        if self.use_structural:
            feats = self._structural_features(words)
            self.struct_mean = feats.mean(axis=0, keepdims=True)
            self.struct_std = feats.std(axis=0, keepdims=True)
            self.struct_std[self.struct_std == 0] = 1

        if self.use_char_svd:
            from sklearn.decomposition import TruncatedSVD

            if self.char_vectorizer is None:
                raise ValueError("char_vectorizer is required when use_char_svd=True")
            char_x = self.char_vectorizer.transform(words)
            n_components = min(self.char_svd_dim, char_x.shape[1] - 1, max(1, char_x.shape[0] - 1))
            self.char_svd = TruncatedSVD(n_components=n_components, random_state=0)
            self.char_svd.fit(char_x)
        return self

    @property
    def dim(self):
        dims = []
        if self.use_semantic:
            dims.append(D)
        if self.use_wordnet:
            if self.wordnet_encoder is None:
                raise ValueError("wordnet_encoder is required when use_wordnet=True")
            dims.append(self.wordnet_encoder.dim)
        if self.use_structural:
            dims.append(len(self._structural_features(["example"])[0]))
        if self.use_phonetic:
            dims.append(len(self._phonetic_features(["example"])[0]))
        if self.use_char_svd:
            dims.append(self.char_svd.n_components if self.char_svd is not None else self.char_svd_dim)
        return sum(dims)

    def encode(self, words, convert_to_tensor=True):
        parts = []
        words = list(words)

        if self.use_semantic:
            if self.word_embedder is None:
                raise ValueError("word_embedder is required when use_semantic=True")
            semantic = self.word_embedder.encode(words, convert_to_tensor=False)
            parts.append(np.asarray(semantic, dtype=np.float32))

        if self.use_wordnet:
            if self.wordnet_encoder is None:
                raise ValueError("wordnet_encoder is required when use_wordnet=True")
            parts.append(self.wordnet_encoder.encode(words))

        if self.use_structural:
            structural = self._structural_features(words)
            if self.struct_mean is not None and self.struct_std is not None:
                structural = (structural - self.struct_mean) / self.struct_std
            parts.append(structural.astype(np.float32))

        if self.use_phonetic:
            parts.append(self._phonetic_features(words).astype(np.float32))

        if self.use_char_svd:
            if self.char_svd is None:
                raise ValueError("Call fit() before encode() when use_char_svd=True")
            char_x = self.char_vectorizer.transform(words)
            parts.append(self.char_svd.transform(char_x).astype(np.float32))

        features = np.concatenate(parts, axis=1)
        if convert_to_tensor:
            return torch.tensor(features, dtype=torch.float)
        return features

    def _structural_features(self, words):
        suffixes = ["s", "ed", "ing", "er", "est", "ly", "tion", "ness", "able", "less"]
        rows = []
        for word in words:
            w = str(word).lower()
            letters = [c for c in w if c.isalpha()]
            vowels = sum(c in "aeiou" for c in letters)
            consonants = sum(c.isalpha() and c not in "aeiou" for c in letters)
            row = [
                len(w),
                len(letters),
                vowels,
                consonants,
                vowels / max(1, len(letters)),
                int(" " in w),
                int("-" in w),
                int("'" in w),
                int(w == w[::-1]),
                sum(c.isdigit() for c in w),
            ]
            row.extend(int(w.endswith(suffix)) for suffix in suffixes)
            rows.append(row)
        return np.asarray(rows, dtype=np.float32)

    def _soundex(self, word):
        w = "".join(c for c in str(word).upper() if c.isalpha())
        if not w:
            return "Z000"
        codes = {
            **dict.fromkeys("BFPV", "1"),
            **dict.fromkeys("CGJKQSXZ", "2"),
            **dict.fromkeys("DT", "3"),
            "L": "4",
            **dict.fromkeys("MN", "5"),
            "R": "6",
        }
        first = w[0]
        encoded = []
        prev = codes.get(first, "")
        for char in w[1:]:
            code = codes.get(char, "")
            if code and code != prev:
                encoded.append(code)
            prev = code
        return (first + "".join(encoded) + "000")[:4]

    def _phonetic_features(self, words):
        rows = []
        for word in words:
            code = self._soundex(word)
            first_letter = np.zeros(26, dtype=np.float32)
            if "A" <= code[0] <= "Z":
                first_letter[ord(code[0]) - ord("A")] = 1

            digits = np.zeros(18, dtype=np.float32)
            for pos, digit in enumerate(code[1:]):
                if digit.isdigit() and 1 <= int(digit) <= 6:
                    digits[pos * 6 + int(digit) - 1] = 1

            rows.append(np.concatenate([first_letter, digits]))
        return np.asarray(rows, dtype=np.float32)


In [ ]:
class YellowConnectionsDataset(Dataset):
    """One state per yellow group and pair of non-yellow distractor groups.

    The source CSV rows are purple, blue, green, yellow. Each state retains
    yellow as the final four rows, giving a shared candidate-label vector;
    models do not receive positional embeddings.
    """

    COLORS_BY_ROW = COLORS_BY_ROW
    TARGET_ROW = TARGET_ROW

    def __init__(self, puzzle_df, word_emb_model, char_vectorizer, feature_builder=None):
        self.puzzle_list = []
        self.feature_builder = feature_builder
        groups_per_date = puzzle_df.groupby("date")["connections"].agg(list)

        for date, group_list in groups_per_date.items():
            if len(group_list) != 4 or any(len(group) != 4 for group in group_list):
                raise ValueError(f"Puzzle {date} must have four groups of four")
            full_words = list(itertools.chain.from_iterable(group_list))
            if len(set(full_words)) != 16:
                raise ValueError(f"Puzzle {date} repeats a word")
            if self.feature_builder is None:
                full_x = word_emb_model.encode(full_words, convert_to_tensor=True).float().cpu()
            else:
                full_x = self.feature_builder.encode(full_words, convert_to_tensor=True).float().cpu()
            edge_builder = getattr(self.feature_builder, "edge_builder", None)
            full_edges = edge_builder.encode(full_words, full_x) if edge_builder is not None else None

            distractor_rows = tuple(row for row in range(4) if row != self.TARGET_ROW)
            for distractors in itertools.combinations(distractor_rows, 2):
                selected_rows = (*distractors, self.TARGET_ROW)
                positions = [4 * row + offset for row in selected_rows for offset in range(4)]
                word_list = [word for row in selected_rows for word in group_list[row]]
                state = {
                    "x": full_x[positions],  # (12, feature_dim)
                    "x_char": torch.tensor(char_vectorizer.transform(word_list).toarray()),
                    "word_list": word_list,
                    "date": date,
                    "target_color": TARGET_COLOR,
                    "distractor_colors": tuple(self.COLORS_BY_ROW[row] for row in distractors),
                }
                if full_edges is not None:
                    state["edge_features"] = full_edges[positions][:, positions]  # (12, 12, edge_dim)
                self.puzzle_list.append(state)

    def __len__(self):
        return len(self.puzzle_list)

    def __getitem__(self, index):
        return self.puzzle_list[index]


# Date split and yellow labels

In [ ]:
# Keep the original date-level split and fit vocabulary-only transforms on train dates.
dates = connections_df["date"].unique()
train_idx = random.sample(range(len(dates)), k=int(len(dates) * 0.8))
train_dates = dates[train_idx]
heldout_dates = np.random.permutation(dates[~np.isin(range(len(dates)), train_idx)])
split_idx = len(heldout_dates) // 2
test_dates = heldout_dates[:split_idx]
val_dates = heldout_dates[split_idx:]
train_vocab_words = list(itertools.chain.from_iterable(
    connections_df.loc[connections_df["date"].isin(train_dates), "connections"]
))
vectorizer.fit(train_vocab_words)
print(f"Dates: train={len(train_dates)}, validation={len(val_dates)}, test={len(test_dates)}")


# Yellow-target data and training

In [ ]:
def make_yellow_loader(puzzle_df, word_embedder, vectorizer, batch_size, shuffle, feature_builder):
    dataset = YellowConnectionsDataset(
        puzzle_df, word_embedder, vectorizer, feature_builder=feature_builder
    )
    return dataset, DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_keep_wordlists
    )


def make_yellow_artifacts(device=None):
    group_idx, relevance = construct_yellow_labels()
    if device is not None:
        group_idx, relevance = group_idx.to(device), relevance.to(device)
    return {
        "idx": group_idx,
        "y": relevance,
        "n": torch.tensor(len(relevance), dtype=torch.long, device=device),
        "k": 1,
    }


def topk_overlap(pred_scores, relevance, k=1):
    predicted = torch.topk(pred_scores, k, dim=1).indices
    target = torch.topk(relevance, k, dim=1).indices
    overlap = (predicted.unsqueeze(2) == target.unsqueeze(1)).any(dim=2).float().sum(dim=1)
    return (overlap / k).mean()


def evaluate_yellow(model, loader, artifact, loss_fn, device):
    model.eval()
    total_loss = total_ndcg = total_top1 = total_mrr = 0.0
    total_examples = 0
    date_correct = {}
    with torch.inference_mode():
        for batch in loader:
            x = batch["x"].to(device)
            batch_size = x.shape[0]
            scores = score_batch(model, x, batch, group_idx=artifact["idx"])
            relevance = artifact["y"].unsqueeze(0).expand(batch_size, -1)
            n = artifact["n"].expand(batch_size)
            flat_scores = scores.squeeze(-1)
            target_index = int(torch.nonzero(artifact["y"], as_tuple=False).item())
            ranks = (flat_scores > flat_scores[:, target_index:target_index + 1]).sum(dim=1) + 1
            correct = ranks == 1
            total_loss += loss_fn(scores, relevance, n).mean().item() * batch_size
            total_ndcg += ndcg(scores, relevance, n, k=1).mean().item() * batch_size
            total_top1 += correct.float().sum().item()
            total_mrr += ranks.float().reciprocal().sum().item()
            for date, solved in zip(batch["date"], correct.cpu().tolist()):
                date_correct.setdefault(date, []).append(solved)
            total_examples += batch_size
    if total_examples == 0:
        raise ValueError("Cannot evaluate an empty yellow loader")
    return {
        "loss": total_loss / total_examples,
        "ndcg_at_1": total_ndcg / total_examples,
        "top1": total_top1 / total_examples,
        "mrr": total_mrr / total_examples,
        "all_distractor_pairs_top1": sum(all(values) for values in date_correct.values()) / len(date_correct),
        "examples": total_examples,
        "dates": len(date_correct),
    }


def format_yellow_metrics(prefix, metrics):
    return (
        f"{prefix} loss: {metrics['loss']:.4f} | "
        f"{prefix} top-1: {metrics['top1']:.1%} | "
        f"{prefix} MRR: {metrics['mrr']:.4f} | "
        f"{prefix} all distractor pairs top-1: {metrics['all_distractor_pairs_top1']:.1%} "
        f"({metrics['dates']} dates)"
    )


def train_yellow(model, train_loader, val_loader, artifact, *, epochs, device, lr, weight_decay, trial=None):
    model = model.to(device)
    loss_fn = LambdaNDCGLoss2().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=5)
    scaler = torch.amp.GradScaler(device)
    artifact = {key: value.to(device) if torch.is_tensor(value) else value for key, value in artifact.items()}
    best_top1, best_state = -float("inf"), None

    for epoch in range(epochs):
        model.train()
        train_loss = train_top1 = 0.0
        for batch in train_loader:
            x = batch["x"].to(device)
            batch_size = x.shape[0]
            relevance = artifact["y"].unsqueeze(0).expand(batch_size, -1)
            n = artifact["n"].expand(batch_size)
            optimizer.zero_grad(set_to_none=True)
            scores = score_batch(model, x, batch, group_idx=artifact["idx"])
            loss = loss_fn(scores, relevance, n).mean()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            train_top1 += topk_overlap(scores.squeeze(-1), relevance, k=1).item()

        train_loss /= len(train_loader)
        train_top1 /= len(train_loader)
        val_metrics = evaluate_yellow(model, val_loader, artifact, loss_fn, device)
        scheduler.step(val_metrics["loss"])
        if val_metrics["top1"] > best_top1:
            best_top1 = val_metrics["top1"]
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        if trial is not None:
            trial.report(val_metrics["top1"], epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
        if epoch % 5 == 0 or epoch == epochs - 1:
            print(
                f"Epoch {epoch} | Train loss: {train_loss:.4f} | Train top-1: {train_top1:.1%} | "
                + format_yellow_metrics("Val", val_metrics)
            )
    model.load_state_dict(best_state)
    return model.to("cpu")


In [ ]:
BATCH_SIZE = 16
BASELINE_EPOCHS = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# The conditioned WordNet node features and graph edges are fit from each full
# 16-word puzzle before yellow states are constructed.
wordnet_encoder = WordNetFeatureEncoder(word_embedder, max_senses=3, puzzle_conditioned=True)
base_feature_builder = WordFeatureBuilder(
    word_embedder=word_embedder, wordnet_encoder=wordnet_encoder, use_wordnet=True
)
graph_edge_builder = PuzzleEdgeBuilder(wordnet_encoder, D)
base_feature_builder.edge_builder = graph_edge_builder

yellow_artifacts = make_yellow_artifacts()
train_yellow_dataset, train_yellow_loader = make_yellow_loader(
    connections_df.loc[connections_df["date"].isin(train_dates)],
    word_embedder, vectorizer, BATCH_SIZE, True, base_feature_builder,
)
val_yellow_dataset, val_yellow_loader = make_yellow_loader(
    connections_df.loc[connections_df["date"].isin(val_dates)],
    word_embedder, vectorizer, BATCH_SIZE, False, base_feature_builder,
)
test_yellow_dataset, test_yellow_loader = make_yellow_loader(
    connections_df.loc[connections_df["date"].isin(test_dates)],
    word_embedder, vectorizer, BATCH_SIZE, False, base_feature_builder,
)
print(
    f"Yellow states: train={len(train_yellow_dataset)}, "
    f"val={len(val_yellow_dataset)}, test={len(test_yellow_dataset)}; "
    f"candidates/state={len(yellow_artifacts['y'])}"
)


## Baseline models

These use the same yellow-target states. They provide non-graph comparisons; the graph model below is tuned separately.


In [ ]:
baseline_model = Baseline(
    in_dim=base_feature_builder.dim, semantic_dim=D, wordnet_dim=wordnet_encoder.dim,
    group_idx=yellow_artifacts["idx"], layers=[512, 512, 512],
).to(device)
baseline_model = train_yellow(
    baseline_model, train_yellow_loader, val_yellow_loader, yellow_artifacts,
    epochs=BASELINE_EPOCHS, device=device, lr=1e-4, weight_decay=1e-5,
)
cpu_artifacts = make_yellow_artifacts(device=torch.device("cpu"))
ranking_loss = LambdaNDCGLoss2().to("cpu")
baseline_val_metrics = evaluate_yellow(baseline_model, val_yellow_loader, cpu_artifacts, ranking_loss, torch.device("cpu"))
baseline_test_metrics = evaluate_yellow(baseline_model, test_yellow_loader, cpu_artifacts, ranking_loss, torch.device("cpu"))
print(format_yellow_metrics("Baseline validation", baseline_val_metrics))
print(format_yellow_metrics("Baseline test", baseline_test_metrics))


In [ ]:
attention_model = AttentionModel(
    in_dim=base_feature_builder.dim, semantic_dim=D, wordnet_dim=wordnet_encoder.dim,
    group_idx=yellow_artifacts["idx"], layers=[1024, 512, 256], attn_layers=5,
).to(device)
attention_model = train_yellow(
    attention_model, train_yellow_loader, val_yellow_loader, yellow_artifacts,
    epochs=BASELINE_EPOCHS, device=device, lr=5e-5, weight_decay=1e-5,
)
attention_val_metrics = evaluate_yellow(attention_model, val_yellow_loader, cpu_artifacts, ranking_loss, torch.device("cpu"))
attention_test_metrics = evaluate_yellow(attention_model, test_yellow_loader, cpu_artifacts, ranking_loss, torch.device("cpu"))
print(format_yellow_metrics("Attention validation", attention_val_metrics))
print(format_yellow_metrics("Attention test", attention_test_metrics))


## Tuned graph attention

The target is the exact yellow group. Optuna selects on validation top-1 accuracy; the held-out test set is evaluated once after selection. The edge-bias control remains available through `edge_bias_scale=0`.


In [ ]:
OPTUNA_TRIALS = 12
OPTUNA_EPOCHS = 20
TUNING_SEED = 0
trial_model_states = {}
trial_model_configs = {}

def objective(trial):
    random.seed(TUNING_SEED)
    np.random.seed(TUNING_SEED)
    torch.manual_seed(TUNING_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(TUNING_SEED)
    model_config = {
        "edge_dim": graph_edge_builder.dim,
        "edge_bias_scale": trial.suggest_categorical("edge_bias_scale", [0.0, 0.25, 0.5, 1.0, 2.0]),
        "in_dim": base_feature_builder.dim,
        "semantic_dim": D,
        "wordnet_dim": wordnet_encoder.dim,
        "lin_layers": trial.suggest_int("lin_layers", 2, 3),
        "lin_width": trial.suggest_categorical("lin_width", [512, 768, 1024]),
        "attn_layers": 1,
        "attn_width": trial.suggest_categorical("attn_width", [256, 384, 512]),
        "attn_heads": 8,
        "dropout": trial.suggest_float("dropout", 0.1, 0.25),
    }
    model = GraphSetTransformer(group_idx=yellow_artifacts["idx"], **model_config).to(device)
    model = train_yellow(
        model, train_yellow_loader, val_yellow_loader, yellow_artifacts,
        epochs=OPTUNA_EPOCHS, device=device,
        lr=trial.suggest_float("lr", 1e-5, 2e-4, log=True),
        weight_decay=trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        trial=trial,
    )
    metrics = evaluate_yellow(model, val_yellow_loader, cpu_artifacts, ranking_loss, torch.device("cpu"))
    trial.set_user_attr("validation_metrics", metrics)
    trial_model_states[trial.number] = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    trial_model_configs[trial.number] = model_config
    model.to("cpu")
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return metrics["top1"]

study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=TUNING_SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=5),
)
study.enqueue_trial({"edge_bias_scale": 0.0})
study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

best_trial = study.best_trial
best_model_config = trial_model_configs[best_trial.number]
best_training_config = {
    "lr": best_trial.params["lr"],
    "weight_decay": best_trial.params["weight_decay"],
    "selection_metric": "validation_top1",
}
model = GraphSetTransformer(group_idx=yellow_artifacts["idx"], **best_model_config)
model.load_state_dict(trial_model_states[best_trial.number])
model = model.to("cpu").eval()
val_metrics = evaluate_yellow(model, val_yellow_loader, cpu_artifacts, ranking_loss, torch.device("cpu"))
test_metrics = evaluate_yellow(model, test_yellow_loader, cpu_artifacts, ranking_loss, torch.device("cpu"))
print("Best validation metrics:", best_trial.user_attrs["validation_metrics"])
print(format_yellow_metrics("Final validation", val_metrics))
print(format_yellow_metrics("Final test", test_metrics))
optuna_results = study.trials_dataframe()
optuna_results


## Save yellow-target graph weights

This checkpoint scores 12-word yellow-target states only. It is not compatible with the 16-word partition-search harness.


In [ ]:
from datetime import datetime

weights_dir = Path("weights")
weights_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
weights_path = weights_dir / f"yellow_target_graph_optuna_{timestamp}.pt"
checkpoint = {
    "model_class": "GraphSetTransformer",
    "model_config": best_model_config,
    "model_state_dict": {key: value.detach().cpu() for key, value in model.state_dict().items()},
    "embedding_model_name": EMBEDDING_MODEL,
    "wordnet_max_senses": wordnet_encoder.max_senses,
    "wordnet_puzzle_conditioned": wordnet_encoder.puzzle_conditioned,
    "edge_feature_names": list(PuzzleEdgeBuilder.feature_names),
    "target_spec": {
        "target_color": TARGET_COLOR,
        "colors_by_row": list(COLORS_BY_ROW),
        "state_words": STATE_WORDS,
        "distractor_pairs_per_puzzle": 3,
    },
    "split_dates": {
        name: pd.to_datetime(values).strftime("%Y-%m-%d").tolist()
        for name, values in (("train", train_dates), ("val", val_dates), ("test", test_dates))
    },
    "dataset_path": str(DATASET_PATH.resolve()),
}
torch.save(checkpoint, weights_path)
print(f"Saved yellow-target graph checkpoint to {weights_path}")
